In [1]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Cell 2: Load Cleaned Data
df = pd.read_csv("../data/sales_data_cleaned.csv")
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Store ID', 'Product ID', 'Date'])

print("Data Loaded:", df.shape)

Data Loaded: (76000, 26)


In [3]:
# Cell 3: Time-based Features
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['Weekday'] = df['Date'].dt.weekday
df['Is_Weekend'] = df['Weekday'].isin([5,6]).astype(int)
df['Quarter'] = df['Date'].dt.quarter
df['Day_of_Year'] = df['Date'].dt.dayofyear

In [4]:
# Cell 4: Lag Features (Very Important for Demand Forecasting)
group_cols = ['Store ID', 'Product ID']

for lag in [1, 3, 7, 14, 30]:
    df[f'Demand_Lag_{lag}'] = df.groupby(group_cols)['Demand'].shift(lag)
    
# Rolling Statistics
for window in [3, 7, 14, 30]:
    df[f'Demand_Rolling_Mean_{window}'] = df.groupby(group_cols)['Demand'].transform(lambda x: x.rolling(window, min_periods=1).mean())
    df[f'Demand_Rolling_Std_{window}'] = df.groupby(group_cols)['Demand'].transform(lambda x: x.rolling(window, min_periods=1).std())

In [5]:
# Cell 5: Consumer Behavior & Marketing Features
df['Price_Elasticity_Proxy'] = df['Demand'] / (df['Net_Price'] + 1)   # Rough proxy
df['Promotion_Effectiveness'] = df['Promotion'] * df['Demand']
df['Competitor_Price_Gap'] = df['Competitor Pricing'] - df['Net_Price']
df['High_Discount'] = (df['Discount'] >= 15).astype(int)

# Interaction Features
df['Promo_Weekend'] = df['Promotion'] * df['Is_Weekend']
df['Weather_Promotion'] = df['Weather Condition'].astype(str) + "_" + df['Promotion'].astype(str)
df['Season_Promotion'] = df['Seasonality'].astype(str) + "_" + df['Promotion'].astype(str)

In [6]:
# Cell 6: Target Encoding for Categories (Good for ML models)
cat_cols = ['Category', 'Region', 'Weather Condition', 'Seasonality']

for col in cat_cols:
    mean_encoding = df.groupby(col)['Demand'].mean()
    df[f'{col}_Target_Enc'] = df[col].map(mean_encoding)

In [8]:
# Cell 7: Final Check & Save
print("Total Features now:", df.shape[1])
print("\nNew Features Created:")
print([col for col in df.columns if col not in ['Date', 'Store ID', 'Product ID']][-15:])  # Last 15 features

# Save engineered dataset
df.to_csv("../data/processed/sales_data_engineered.csv", index=False)
print("\n✅ Feature Engineering Completed & Saved!")

Total Features now: 53

New Features Created:
['Demand_Rolling_Mean_14', 'Demand_Rolling_Std_14', 'Demand_Rolling_Mean_30', 'Demand_Rolling_Std_30', 'Price_Elasticity_Proxy', 'Promotion_Effectiveness', 'Competitor_Price_Gap', 'High_Discount', 'Promo_Weekend', 'Weather_Promotion', 'Season_Promotion', 'Category_Target_Enc', 'Region_Target_Enc', 'Weather Condition_Target_Enc', 'Seasonality_Target_Enc']

✅ Feature Engineering Completed & Saved!
